In [2]:
# -*- coding: utf-8 -*-
import sys
import os
import glob
import gc
import json
import numpy as np
from tqdm import tqdm
from scipy.stats import gaussian_kde

# ==============================================================================
# PERTAHANAN MURNI CPU & ANTI-LEAK UNTUK MACBOOK M3 PRO
# ==============================================================================
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU') 
from tensorflow import keras
from tensorflow.keras import backend as K
# ==============================================================================

# --- KONFIGURASI PATH UTAMA ---
BASE_REP = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo'
if BASE_REP not in sys.path:
    sys.path.append(BASE_REP)

from Library import utils, dataset

# Direktori Sumber Data Master 3C Indonesia
DIR_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_gempa_9s"
DIR_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_noise_9s"

def predict_and_evaluate_pure_3c_batch(model, b_E, b_N, b_Z, batch_labels, kde_noise_3c, kde_le_3c):
    """
    Evaluasi 3C Murni sesuai Protokol Zhi Geng (Fig. 4e):
    Menggabungkan fitur ekstraksi 3 sumbu menjadi matriks koordinat 3D,
    laku dievaluasi langsung ke fungsi densitas bersama tanpa operasi Late Fusion.
    """
    num_points = 700
    in_E = np.array(b_E, dtype=np.float32).reshape(-1, num_points, 1)
    in_N = np.array(b_N, dtype=np.float32).reshape(-1, num_points, 1)
    in_Z = np.array(b_Z, dtype=np.float32).reshape(-1, num_points, 1)
    
    with tf.device('/CPU:0'):
        pred_E = model.predict_on_batch(in_E)
        pred_N = model.predict_on_batch(in_N)
        pred_Z = model.predict_on_batch(in_Z)
        
    # MURNI ZHI GENG: Satukan koordinat ruang laten menjadi array 3D [E, N, Z] (Batch, 3)
    joint_embeddings = np.hstack([pred_E, pred_N, pred_Z])
    
    # Evaluasi simultan ke kurva multi-variat kepadatan bersama UUSS
    like_noise = kde_noise_3c.pdf(joint_embeddings.T)
    like_le = kde_le_3c.pdf(joint_embeddings.T)
    
    likelihoods = np.vstack([like_noise, like_le])
    preds = np.argmax(likelihoods, axis=0)
    
    local_tp, local_tn, local_fp, local_fn = 0, 0, 0, 0
    for t_lbl, p_lbl in zip(batch_labels, preds):
        if t_lbl == 1 and p_lbl == 1: local_tp += 1
        elif t_lbl == 0 and p_lbl == 0: local_tn += 1
        elif t_lbl == 0 and p_lbl == 1: local_fp += 1
        elif t_lbl == 1 and p_lbl == 0: local_fn += 1
        
    return local_tp, local_tn, local_fp, local_fn

if __name__ == "__main__":
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    # --------------------------------------------------------------------------
    # FASE 0: LOAD MODEL & PEMULIHAN FUNGSI PDF 3D BERFUNGSI (KOREKSI ORIENTASI)
    # --------------------------------------------------------------------------
    print("[INFO] Memuat Model dan Memulihkan Fungsi Kepadatan 3C Bersama...")
    with tf.device('/CPU:0'):
        embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    emb_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
    emb_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
    emb_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    
    keys = list(emb_Z.keys())
    kn = next((k for k in keys if k.lower() in ['noise', 'no']), keys[0])
    kle = next((k for k in keys if k.lower() in ['le', 'earthquake', 'eq']), keys[1] if len(keys)>1 else keys[0])

    # KUNCI RECOVERY: Sinkronisasi pemanggilan array sesuai bentuk ekstraksi repositori asli
    # Memastikan output akhir bertipe (3, N) dengan melakukan konversi array murni transpus
    val_E_kn = np.array(emb_E[kn]).reshape(1, -1)
    val_N_kn = np.array(emb_N[kn]).reshape(1, -1)
    val_Z_kn = np.array(emb_Z[kn]).reshape(1, -1)
    
    val_E_le = np.array(emb_E[kle]).reshape(1, -1)
    val_N_le = np.array(emb_N[kle]).reshape(1, -1)
    val_Z_le = np.array(emb_Z[kle]).reshape(1, -1)

    train_noise_3c = np.vstack([val_E_kn, val_N_kn, val_Z_kn]) 
    train_le_3c = np.vstack([val_E_le, val_N_le, val_Z_le])

    # Membangun fungsi kepadatan multi-variat 3D murni penemu yang terbukti berfungsi
    kde_noise_3c = gaussian_kde(train_noise_3c)
    kde_le_3c = gaussian_kde(train_le_3c)
    
    del emb_E, emb_N, emb_Z, val_E_kn, val_N_kn, val_Z_kn, val_E_le, val_N_le, val_Z_le, train_noise_3c, train_le_3c
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 1: PERSIAPAN DATASET UJI INDONESIA
    # --------------------------------------------------------------------------
    files_gempa = glob.glob(os.path.join(DIR_GEMPA, "*.npy"))
    files_noise = glob.glob(os.path.join(DIR_NOISE, "*.npy"))
    
    print(f"[INFO] Total Data Uji Indonesia Terdeteksi: {len(files_gempa):,} Gempa | {len(files_noise):,} Noise")
    
    all_files = [(f, 1) for f in files_gempa] + [(f, 0) for f in files_noise]
    np.random.seed(42)
    np.random.shuffle(all_files)
    
    file_paths = [item[0] for item in all_files]
    file_labels = [item[1] for item in all_files]
    
    del all_files, files_gempa, files_noise
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: INFERENSI TERMINAL 3C MURNI INDONESIA (PERSIS ZHI GENG)
    # --------------------------------------------------------------------------
    num_points = 700
    BUFFER_SIZE = 128
    buffer_E, buffer_N, buffer_Z = [], [], []
    buffer_labels = []

    TP, TN, FP, FN = 0, 0, 0, 0
    FILE_KORUP = 0

    print(f"\n[INFO] Memulai eksekusi 3C murni Indonesia untuk {len(file_paths):,} sampel...")
    
    for idx in tqdm(range(len(file_paths)), desc="Indonesia Pure 3C 100K"):
        try:
            wave_3c = np.load(file_paths[idx])
            
            if not np.isfinite(wave_3c).all() or wave_3c.shape != (700, 3):
                FILE_KORUP += 1
                continue

            # MURNI ZHI GENG: Slicing tiga komponen spasial (0: E, 1: N, 2: Z)
            wave_3c_window = wave_3c
            true_label = file_labels[idx]
            
            # PRE-PROCESSING MURNI: Detrending & Normalisasi per Komponen (Tanpa Filter)
            e_comp = wave_3c_window[:, 0] - np.mean(wave_3c_window[:, 0])
            norm_e = np.max(np.abs(e_comp))
            if norm_e > 0: e_comp /= norm_e
            
            n_comp = wave_3c_window[:, 1] - np.mean(wave_3c_window[:, 1])
            norm_n = np.max(np.abs(n_comp))
            if norm_n > 0: n_comp /= norm_n
            
            z_comp = wave_3c_window[:, 2] - np.mean(wave_3c_window[:, 2])
            norm_z = np.max(np.abs(z_comp))
            if norm_z > 0: z_comp /= norm_z
            
            buffer_E.append(e_comp)
            buffer_N.append(n_comp)
            buffer_Z.append(z_comp)
            buffer_labels.append(true_label)
            
            if len(buffer_Z) == BUFFER_SIZE:
                b_tp, b_tn, b_fp, b_fn = predict_and_evaluate_pure_3c_batch(
                    embedding_model, buffer_E, buffer_N, buffer_Z, buffer_labels,
                    kde_noise_3c, kde_le_3c
                )
                TP += b_tp; TN += b_tn; FP += b_fp; FN += b_fn
                
                buffer_E.clear(); buffer_N.clear(); buffer_Z.clear()
                buffer_labels.clear()
                
            if (idx + 1) % 10000 == 0:
                print(f"\n[LOG 3C INDO PURE INTERIM {idx+1}] TP:{TP} \u007c TN:{TN} \u007c FP:{FP} \u007c FN:{FN}")
                gc.collect() 
                K.clear_session()
                
        except Exception:
            FILE_KORUP += 1
            continue

    if len(buffer_Z) > 0:
        b_tp, b_tn, b_fp, b_fn = predict_and_evaluate_pure_3c_batch(
            embedding_model, buffer_E, buffer_N, buffer_Z, buffer_labels,
            kde_noise_3c, kde_le_3c
        )
        TP += b_tp; TN += b_tn; FP += b_fp; FN += b_fn

    # ==========================================================================
    # FASE 3: KALKULASI AKHIR METRIK REPLIKASI MURNI 3C DATA INDONESIA
    # ==========================================================================
    total_data = TP + TN + FP + FN
    akurasi = (TP + TN) / total_data if total_data > 0 else 0
    recall_tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    spesifisitas_tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    presisi_ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    f1_score = 2 * (presisi_ppv * recall_tpr) / (presisi_ppv + recall_tpr) if (presisi_ppv + recall_tpr) > 0 else 0

    print("\n=======================================================")
    print(" HASIL REPLIKASI MURNI 3C GAYA ZHI GENG (DATA INDONESIA)")
    print("=======================================================")
    print(f"Total Data Terproses  : {total_data:,} sampel 3C")
    print(f"Total File Korup      : {FILE_KORUP:,} file")
    print(f"True Positives (TP)   : {TP:,}")
    print(f"True Negatives (TN)   : {TN:,}")
    print(f"False Positives (FP)  : {FP:,}")
    print(f"False Negatives (FN)  : {FN:,}")
    print("-------------------------------------------------------")
    print(f"Akurasi Global        : {akurasi:.4f} ({(akurasi*100):.2f}%)")
    print(f"Recall (TPR)          : {recall_tpr:.4f} ({(recall_tpr*100):.2f}%)")
    print(f"Spesifisitas (TNR)    : {spesifisitas_tnr:.4f} ({(spesifisitas_tnr*100):.2f}%)")
    print(f"Presisi (PPV)         : {presisi_ppv:.4f} ({(presisi_ppv*100):.2f}%)")
    print(f"F1-Score              : {f1_score:.4f} ({(f1_score*100):.2f}%)")
    print("=======================================================")

[INFO] Memuat Model dan Memulihkan Fungsi Kepadatan 3C Bersama...
[INFO] Total Data Uji Indonesia Terdeteksi: 52,427 Gempa | 52,425 Noise

[INFO] Memulai eksekusi 3C murni Indonesia untuk 104,852 sampel...


Indonesia Pure 3C 100K:   7%|▋         | 7167/104852 [00:03<00:51, 1908.28it/s]


KeyboardInterrupt: 